# 01 Data Understanding

**Objective:** reproduce and document the dataset verification for the Airbnb New York City 5Y dataset, so every reported statistic can be traced to code that runs on the actual files.

**Reference values** (from earlier verification) are used only to *compare* against the values calculated here. They are never used as output.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

locate the data folder

In [2]:
# Works whether the notebook is run from notebooks/ or from the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
for f in sorted(RAW.iterdir()):
    print(f.name, f"{f.stat().st_size / 1e6:.1f} MB")

Project root: c:\Users\LENOVO\airbnb-nyc-spatial-temporal-analysis
.gitkeep 0.0 MB
listings.parquet 2.9 MB
listings_monthly.csv 7.6 MB
nta2020.geojson 4.6 MB
past_rates.csv 24.9 MB
past_rates.parquet 2.0 MB


load the two main files and compare their shapes

In [3]:
listings = pd.read_parquet(RAW / "listings.parquet")
past_rates = pd.read_csv(RAW / "past_rates.csv")

# Reference values from earlier verification (used ONLY for comparison)
reference = {"listings": (4266, 81), "past_rates": (150034, 23)}
computed = {"listings": listings.shape, "past_rates": past_rates.shape}

for name in reference:
    status = "MATCH" if computed[name] == reference[name] else "DIFFERENT - investigate"
    print(f"{name}: computed {computed[name]} | reference {reference[name]} | {status}")

listings: computed (4266, 81) | reference (4266, 81) | MATCH
past_rates: computed (150034, 23) | reference (150034, 23) | MATCH


## 1. Structure: columns and data types
We list the columns and data types of both files. The analysis relies on `listing_id`, `month_date` and `avg_daily_rate` in `past_rates`; this check confirms they exist and shows how they were read.

In [4]:
print("listings columns:", len(listings.columns))
print(listings.dtypes.value_counts().to_string())
print()
print("past_rates columns:", len(past_rates.columns))
print(past_rates.dtypes.to_string())

needed = ["listing_id", "month_date", "avg_daily_rate", "latitude", "longitude", "neighborhood"]
print()
print("Needed columns present in past_rates:", {c: c in past_rates.columns for c in needed})

listings columns: 81
float64    56
str        12
object     12
int64       1

past_rates columns: 23
listing_id                 int64
listing_type                 str
room_type                    str
latitude                 float64
longitude                float64
bedrooms                 float64
country                      str
currency                     str
neighborhood             float64
month_date                   str
available_days             int64
unavailable_days           int64
occupancy                float64
avg_daily_rate           float64
native_avg_daily_rate    float64
revenue                  float64
native_revenue           float64
active                      bool
booking_lead_time_avg    float64
length_of_stay_avg       float64
booked_rate_avg          float64
revpar                   float64
native_revpar            float64

Needed columns present in past_rates: {'listing_id': True, 'month_date': True, 'avg_daily_rate': True, 'latitude': True, 'longitude': True,

## 2. Listing-ID matching
Every listing in `listings` should have rows in `past_rates`. Matching IDs tells us the two files describe the same listings.

In [5]:
ref = {"unique_listings": 4266, "unique_hist_ids": 4266, "matching": 4266, "match_pct": 100.0}

ids_listings = set(listings["listing_id"])
ids_hist = set(past_rates["listing_id"])
matching = ids_listings & ids_hist
match_pct = 100 * len(matching) / len(ids_listings)

calc = {"unique_listings": len(ids_listings), "unique_hist_ids": len(ids_hist),
        "matching": len(matching), "match_pct": round(match_pct, 2)}

for k in ref:
    print(f"{k}: computed {calc[k]} | reference {ref[k]} | {'MATCH' if calc[k] == ref[k] else 'DIFFERENT - investigate'}")

print("IDs in listings but not in past_rates:", len(ids_listings - ids_hist))
print("IDs in past_rates but not in listings:", len(ids_hist - ids_listings))

unique_listings: computed 4266 | reference 4266 | MATCH
unique_hist_ids: computed 4266 | reference 4266 | MATCH
matching: computed 4266 | reference 4266 | MATCH
match_pct: computed 100.0 | reference 100.0 | MATCH
IDs in listings but not in past_rates: 0
IDs in past_rates but not in listings: 0


## 3. Date coverage and records by year
`month_date` must be converted to a real date type before we can extract the year. We then compare the first and last month, the number of unique months, and the records and months available per year with the reference table.

In [7]:
dates = pd.to_datetime(past_rates["month_date"], errors="coerce")
print("Unparseable month_date values:", int(dates.isna().sum()))
print("First observation:", dates.min().strftime("%B %Y"))
print("Last observation:", dates.max().strftime("%B %Y"))
print("Unique months:", dates.dt.to_period("M").nunique())

by_year = (pd.DataFrame({"year": dates.dt.year, "month": dates.dt.month})
           .groupby("year").agg(records=("month", "size"), months_available=("month", "nunique")))

reference_years = pd.DataFrame({"records": {2021: 25050, 2022: 34466, 2023: 37311, 2024: 27006, 2025: 22951, 2026: 3250},
                                "months_available": {2021: 10, 2022: 12, 2023: 12, 2024: 12, 2025: 12, 2026: 2}})

comparison = by_year.join(reference_years.add_suffix("_reference"))
comparison["match"] = ((comparison["records"] == comparison["records_reference"]) &
                       (comparison["months_available"] == comparison["months_available_reference"]))
print()
print(comparison.to_string())

Unparseable month_date values: 0
First observation: March 2021
Last observation: February 2026
Unique months: 60

      records  months_available  records_reference  months_available_reference  match
year                                                                                 
2021    25050                10              25050                          10   True
2022    34466                12              34466                          12   True
2023    37311                12              37311                          12   True
2024    27006                12              27006                          12   True
2025    22951                12              22951                          12   True
2026     3250                 2               3250                           2   True


## 4. Neighbourhood field
The `neighborhood` field is checked in both files. Earlier inspection suggested it holds ZIP-code-like numbers rather than neighbourhood names; this cell documents that from the data itself.

In [8]:
for name, df in [("listings", listings), ("past_rates", past_rates)]:
    if "neighborhood" in df.columns:
        col = df["neighborhood"]
        print(f"{name}: dtype={col.dtype}, unique values={col.nunique()}, missing={int(col.isna().sum())}")
        print("  sample values:", list(col.dropna().astype(str).unique()[:8]))
    else:
        print(f"{name}: no 'neighborhood' column")

print()
print("Other geographic columns in listings:", [c for c in listings.columns if c in ("county", "neighborhood", "latitude", "longitude")])

listings: dtype=str, unique values=195, missing=99
  sample values: ['11222', '11231', '11213', '10027', '10001', '10014', '10026', '10024']
past_rates: dtype=float64, unique values=195, missing=923
  sample values: ['11222.0', '11231.0', '11213.0', '10027.0', '10001.0', '10014.0', '10026.0', '10024.0']

Other geographic columns in listings: ['county', 'latitude', 'longitude', 'neighborhood']


## 5. Basic missing-value and currency checks
We count missing values in the columns the analysis relies on, and check that all rates use a single currency. Only checking here, nothing is removed or changed (cleaning happens in notebook 02).

In [9]:
key_cols = ["listing_id", "month_date", "avg_daily_rate", "latitude", "longitude", "neighborhood", "room_type", "currency", "occupancy"]
missing = past_rates[key_cols].isna().sum().to_frame("missing")
missing["percent"] = (100 * missing["missing"] / len(past_rates)).round(2)
print(missing.to_string())

print()
print("Currency values in past_rates:")
print(past_rates["currency"].value_counts(dropna=False).to_string())

print()
print("avg_daily_rate summary:")
print(past_rates["avg_daily_rate"].describe().round(2).to_string())
print("avg_daily_rate <= 0:", int((past_rates["avg_daily_rate"] <= 0).sum()))

print()
print("Top 10 columns with missing values in listings:")
print(listings.isna().sum().sort_values(ascending=False).head(10).to_string())

                missing  percent
listing_id            0     0.00
month_date            0     0.00
avg_daily_rate        0     0.00
latitude              0     0.00
longitude             0     0.00
neighborhood        923     0.62
room_type             0     0.00
currency              0     0.00
occupancy             0     0.00

Currency values in past_rates:
currency
USD    150034

avg_daily_rate summary:
count    150034.00
mean        188.27
std         190.84
min          15.00
25%          83.50
50%         140.60
75%         225.90
max        4983.10
avg_daily_rate <= 0: 0

Top 10 columns with missing values in listings:
subdivision                4266
county                     4264
instant_book               3810
extra_guest_fee            3063
native_extra_guest_fee     3063
cleaning_fee               2635
native_cleaning_fee        2635
professional_management    2478
registration               2164
cohost                     2164


## Summary
- `listings` has 4,266 rows × 81 columns and `past_rates` has 150,034 rows × 23 columns; both match the reference values.
- All 4,266 listing IDs appear in both files (100% match, none unmatched in either direction).
- `month_date` parses without errors: March 2021 to February 2026, 60 unique months. Yearly record counts match the reference table (2022-2025 are complete years; 2021 and 2026 are partial).
- All rates are in USD. `avg_daily_rate` has no missing values and none ≤ 0; it is right-skewed (median 140.60, mean 188.27, maximum 4,983.10), which supports using the median.
- The `neighborhood` field holds ZIP-code-like values (195 distinct), not neighbourhood names. It is stored as text in `listings` and as a number in `past_rates`, with 99 listings and 923 monthly rows missing. Neighbourhoods will instead be assigned by a spatial join on the coordinates (0 missing).

## Limitations / observations
- Extreme rates have not yet been investigated; this is done in notebook 02.
- Verification here uses `past_rates.csv`; the parquet version stores `neighborhood` with a different data type.